# 10주 · 추정량·모평균 신뢰구간

공학통계학 CE202 · 2027학년도 1학기

슬라이드의 개념·숫자·풀이 순서와 연결한 **새 병행 실습**. 원본 Colab 복원본이 아닌 신규 자료

**목표**: 자료·확률·분포·추정·검정의 원리 이해와 통계적 해석 심화

**수업**: 2027-05-04 추정 (1) · 대면 / 2027-05-06 추정 (2) · 대면

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE202-statistics-for-engineers/week10/CE202_W10.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE202-statistics-for-engineers/week10)

질문·가정 확인 → 손계산·예측 → 계산·시각화 → 한 조건 변경 → 통계적 해석·한계 설명

셀 실행: Shift+Enter. 아래 코드의 None은 직접 채울 항목. 원본과 변경 사본의 결과를 구분

핵심 실습: 손계산·원리와 계산식의 연결·조건 변경·해석. AI 활용은 개념 학습 후 선택 활동

## 0. 실행 환경

필요한 패키지의 최소 버전 확인. 기본 코드와 보조 함수는 이번 주 실습에 필요한 도구. 세부 구현 암기 불필요

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.16", "matplotlib": "3.10", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math, tempfile, base64, zipfile, io
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
WORK = Path(tempfile.mkdtemp(prefix="ce202-"))
rng = np.random.default_rng(2027)

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 수치 자료와 최소 표본수 확인 필요")
    return x

def probabilities(values):
    p = vector(values)
    if np.any(p < 0) or not np.isclose(p.sum(), 1):
        raise ValueError("확률의 비음수성·합계 1 확인 필요")
    return p

def mean_ci(values, confidence=0.95):
    x = vector(values, 2)
    if not 0 < confidence < 1:
        raise ValueError("신뢰수준은 0과 1 사이")
    se = x.std(ddof=1) / np.sqrt(len(x))
    q = stats.t.ppf((1 + confidence) / 2, df=len(x)-1)
    return {"n": len(x), "mean": x.mean(), "se": se,
            "df": len(x)-1, "low": x.mean()-q*se, "high": x.mean()+q*se}

def wilson_interval(successes, n, confidence=0.95):
    if not isinstance(n, (int, np.integer)) or n <= 0 or not 0 <= successes <= n:
        raise ValueError("0 ≤ 성공수 ≤ 양의 정수 표본수 확인 필요")
    if int(successes) != successes or not 0 < confidence < 1:
        raise ValueError("정수 성공수·신뢰수준 확인 필요")
    z = stats.norm.ppf((1+confidence)/2)
    phat = successes / n
    den = 1 + z*z/n
    center = (phat + z*z/(2*n)) / den
    half = z / den * np.sqrt(phat*(1-phat)/n + z*z/(4*n*n))
    return center-half, center+half

def anova_summary(groups):
    arrays = [vector(g, 2) for g in groups]
    if len(arrays) < 2:
        raise ValueError("두 집단 이상 필요")
    n = np.array([len(g) for g in arrays]); means = np.array([g.mean() for g in arrays])
    grand = np.concatenate(arrays).mean()
    ss_between = np.sum(n*(means-grand)**2)
    ss_within = sum(np.sum((g-g.mean())**2) for g in arrays)
    df_between = len(arrays)-1; df_within = n.sum()-len(arrays)
    if ss_within <= 0:
        raise ValueError("집단 내 변동이 0이면 일반적인 F 계산 불가")
    F = (ss_between/df_between) / (ss_within/df_within)
    return {"ss_between": ss_between, "ss_within": ss_within,
            "df_between": df_between, "df_within": df_within,
            "F": F, "p": stats.f.sf(F, df_between, df_within)}

def regression(x, y):
    x = vector(x, 3); y = vector(y, 3)
    if len(x) != len(y) or np.ptp(x) == 0:
        raise ValueError("같은 길이·서로 다른 x 값 확인 필요")
    fit = stats.linregress(x, y)
    fitted = fit.intercept + fit.slope*x
    residual = y-fitted
    sse = np.sum(residual**2); sst = np.sum((y-y.mean())**2)
    if sst <= 0:
        raise ValueError("y 변동이 0이면 일반적인 R² 해석 불가")
    return {"slope": fit.slope, "intercept": fit.intercept, "r2": 1-sse/sst,
            "sse": sse, "mse_residual": sse/(len(x)-2), "mse_prediction": sse/len(x),
            "p": fit.pvalue, "stderr": fit.stderr, "fitted": fitted, "residual": residual}


## S01. 원본 균등분포 MOM과 MLE

**슬라이드 연결**: 2026_S1_Statistics_Wk10a_Inference1.pptx p27, 31, 42

**자료 구분**: 원본 예제·수치 기반

**먼저 예측**: U(0,θ)의 θ가 최대 관측값보다 작아도 자료를 설명할 수 있는가?

계산 전에 분모·단위·조건을 한 줄로 기록

In [ ]:
strength=np.array([3.2,4.5,2.8,5.1,4.0]); theta_mom=2*strength.mean(); theta_mle=strength.max()
print({"MOM":theta_mom,"MLE":theta_mle})
assert np.isclose(theta_mom,7.84) and np.isclose(theta_mle,5.1)
# 지지집합 검토용 변형: MOM이 최대 관측값보다 작을 수도 있음
variant=np.array([1.,1,10]);print("변형 MOM/표본최대:",2*variant.mean(),variant.max())

**조건 변경**: 관측값 사본 하나를 바꾸고 MOM·MLE의 반응 비교

위 코드를 복사한 셀에서 한 조건씩 변경. 원본 셀은 보존

**선택 심화 · AI 검토**: 관측값·지지집합 U(0,θ)·추정 방법을 명시한 계산 요청

**결과 검토**: 가능도는 고정 자료에 대한 모수의 함수. 연속형 한 점의 확률과 구분

예측·손계산·변경 결과·개념 설명·적용 한계 기록

In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## S02. 원본 함유량 6회의 t 신뢰구간

**슬라이드 연결**: 2026_S1_Statistics_Wk10b_Inference2.pptx p9, 20, 23, 29, 30

**자료 구분**: 원본 예제·수치 기반

**먼저 예측**: 고정된 모평균이 이번 구간에 들어갈 확률이 .95라는 뜻인가?

계산 전에 분모·단위·조건을 한 줄로 기록

In [ ]:
content_pct=np.array([.5,1.0,.9,.2,.7,.4]); ci=mean_ci(content_pct)
print(ci)
assert ci["df"]==5 and np.isclose(ci["mean"],3.7/6)
print("z근사와 t 오차한계:",stats.norm.isf(.025)*ci["se"],stats.t.isf(.025,5)*ci["se"])
# 모의실험: 정규·독립 가정 아래 같은 t 구간의 장기 포함률
sim=rng.normal(.6,.3,size=(4000,6)); m=sim.mean(axis=1); se=sim.std(axis=1,ddof=1)/np.sqrt(6)
covered=np.mean((m-stats.t.isf(.025,5)*se<=.6)&(.6<=m+stats.t.isf(.025,5)*se))
print("모의실험 장기 포함률:",covered);assert .92<covered<.98

**조건 변경**: 신뢰수준 .95→.99 변경 시 구간 폭 비교

위 코드를 복사한 셀에서 한 조건씩 변경. 원본 셀은 보존

**선택 심화 · AI 검토**: 원본 함유량 단위 %, n6, 모분산 미지, t구간·정규/독립 가정 명시

**결과 검토**: 모의실험과 원본 관측값 구분. 표본평균의 구간을 개인값 범위로 해석하지 않음

예측·손계산·변경 결과·개념 설명·적용 한계 기록

In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## 주차 확인

MOM·MLE와 t 신뢰구간의 조건 기록

입력 항목:

- `uniform_mle`: 균등분포 U(0,θ) 원본 관측값의 θ 최대가능도 추정값 · 수치
- `chemical_mean`: 원본 화학 함유량 6개 값의 평균. 단위 % 유지 · 수치
- `t_df`: 원본 화학 함유량 t 신뢰구간의 자유도 · 수치
- `confidence_is_posterior_probability`: 빈도주의 95% 신뢰구간이 계산된 뒤, 고정 모수가 그 구간에 있을 사후확률 .95를 뜻하는가? · Python `True`/`False`

공개 확인용 항목. 결과값만 복사한 경우 학습 완료로 판단 불가. 분석 가정·자료 출처·추론 범위는 아래 기록과 함께 검토

In [ ]:
answers = {'uniform_mle': None, 'chemical_mean': None, 't_df': None, 'confidence_is_posterior_probability': None}


In [ ]:
def check_answers(values):
    expected = {'uniform_mle': 5.1, 'chemical_mean': 0.6166666666666667, 't_df': 5, 'confidence_is_posterior_probability': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## 원리 탐구·결과 해석 기록

1. 문제·자료·단위·모형과 가정
2. 실행 전 예측·작은 손계산과 근거
3. 기준 계산의 중간값·결과와 원리의 연결
4. 한 조건 변경 결과와 차이의 원인
5. 독립 검산·적용 조건·불확실성·한계
6. 자신의 말로 설명할 핵심 원리 한 가지

AI 활용은 선택. 사용 시 요청·의심한 출력·수정·검산 근거를 별도 기록. 글자 수 자동 채점 없음

In [ ]:
review_log = {
    "문제·자료·단위·가정": None,
    "예측·손계산 근거": None,
    "중간 계산·원리 연결": None,
    "조건 변경·차이의 원인": None,
    "검산·해석·적용 한계": None,
    "내 설명": None,
}
# 선택 활동: AI 활용 시 요청·의심한 출력·수정·검산 근거를 별도 기록
